In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

folder = Path("../output_csv")

input_file = folder / "residential_cleaned_preprocessed.csv"

df = pd.read_csv(input_file, low_memory=False)

df.head()

,ClosePrice,source_month,LivingArea,DaysOnMarket,LotSizeSquareFeet,YearBuilt,BathroomsTotalInteger,BedroomsTotal,GarageSpaces,Latitude,...,PostalCode_92563,PostalCode_92584,PostalCode_92592,PostalCode_92596,PostalCode_93065,PostalCode_93535,PostalCode_93536,PostalCode_93551,PostalCode_94513,PostalCode_Other
0,1800000.0,202505,1.448510,-0.753592,-0.020607,0.984604,1.229861,1.602918,0.306167,-0.475161,...,False,False,False,False,False,False,False,False,False,True
1,1200000.0,202505,-0.853308,-0.753592,-0.020792,-1.299200,-0.561218,-1.572704,-0.001975,-0.363308,...,False,False,False,False,False,False,False,False,False,True
2,2250000.0,202505,-0.880343,-0.753592,-0.020726,-0.755437,-1.456758,-0.514163,-0.310118,1.467452,...,False,False,False,False,False,False,False,False,False,True
3,1425000.0,202505,-0.360889,-0.753592,-0.020622,0.440841,-0.561218,-0.514163,-0.001975,-0.483640,...,False,False,False,False,False,False,False,False,False,True
4,660000.0,202505,-0.733583,-0.753592,-0.020800,0.368339,0.334321,-0.514163,-0.001975,-0.890424,...,False,False,False,False,False,False,False,False,False,True


In [2]:
target = 'ClosePrice'
# Train/test split by most recent month
df['source_month'] = df['source_month'].astype(float).astype(int).astype(str)

months = sorted(df['source_month'].unique())

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

model_results = []

drop_cols = [target, 'source_month']
feature_cols = [col for col in df.columns if col not in drop_cols]

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

for X_window in [3, 6, 9, 12]:
    train_months = months[-(X_window + 1):-1]
    test_month = months[-1]

    train_df = df[
        df["source_month"].isin(train_months)
    ].copy()

    test_df = df[
        df["source_month"] == test_month
    ].copy()

    X_train = train_df[feature_cols]
    y_train = train_df[target]

    X_test = test_df[feature_cols]
    y_test = test_df[target]

    for model_name, model in models.items():

        model.fit(X_train, y_train)

        train_preds = model.predict(X_train)
        test_preds = model.predict(X_test)

        train_r2 = r2_score(y_train, train_preds)
        test_r2 = r2_score(y_test, test_preds)

        mae = mean_absolute_error(
            y_test,
            test_preds
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                test_preds
            )
        )

        model_results.append({
            "model": model_name,
            "training_window_months": X_window,
            "train_months": ", ".join(train_months),
            "test_month": test_month,
            "train_r2": train_r2,
            "test_r2": test_r2,
            "mae": mae,
            "rmse": rmse
        })

comparison_df = pd.DataFrame(model_results)

comparison_df.sort_values(
    by="test_r2",
    ascending=False
)

,model,training_window_months,train_months,test_month,train_r2,test_r2,mae,rmse
9,Linear Regression,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,0.019138,0.359520,537784.982295,1.343528e+06
6,Linear Regression,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,0.014867,0.350591,531346.068076,1.352861e+06
3,Linear Regression,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,0.020179,0.333577,553658.476867,1.370468e+06
0,Linear Regression,3,"202602, 202603, 202604",202605,0.016948,0.253278,625859.300317,1.450686e+06
5,Random Forest,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,0.826433,-3.760875,332385.332041,3.663003e+06
11,Random Forest,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,0.843407,-7.892013,350976.583701,5.006036e+06
8,Random Forest,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,0.838976,-10.985011,386612.533633,5.811833e+06
2,Random Forest,3,"202602, 202603, 202604",202605,0.872692,-11.768353,459414.058827,5.998758e+06
4,Decision Tree,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,0.999973,-15.222809,412118.155673,6.761716e+06
1,Decision Tree,3,"202602, 202603, 202604",202605,1.000000,-41.429706,542636.665221,1.093525e+07


In [5]:
# Find the best Linear Regression baseline
best_baseline_row = (
    comparison_df[
        comparison_df["model"] == "Linear Regression"
    ]
    .sort_values(
        by="test_r2",
        ascending=False
    )
    .iloc[0]
)

best_baseline_r2 = best_baseline_row["test_r2"]
best_baseline_window = best_baseline_row["training_window_months"]

print("Best baseline window:", best_baseline_window)
print(f"Best baseline test R²: {best_baseline_r2:.4f}")

Best baseline window: 12
Best baseline test R²: 0.3595


In [6]:
comparison_df["baseline_test_r2"] = best_baseline_r2

comparison_df["r2_change_from_baseline"] = (
    comparison_df["test_r2"]
    - comparison_df["baseline_test_r2"]
)

comparison_df.sort_values(
    by="test_r2",
    ascending=False
)

,model,training_window_months,train_months,test_month,train_r2,test_r2,mae,rmse,baseline_test_r2,r2_change_from_baseline
9,Linear Regression,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,0.019138,0.359520,537784.982295,1.343528e+06,0.35952,0.000000
6,Linear Regression,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,0.014867,0.350591,531346.068076,1.352861e+06,0.35952,-0.008929
3,Linear Regression,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,0.020179,0.333577,553658.476867,1.370468e+06,0.35952,-0.025942
0,Linear Regression,3,"202602, 202603, 202604",202605,0.016948,0.253278,625859.300317,1.450686e+06,0.35952,-0.106242
5,Random Forest,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,0.826433,-3.760875,332385.332041,3.663003e+06,0.35952,-4.120395
11,Random Forest,12,"202505, 202506, 202507, 202508, 202509, 202510...",202605,0.843407,-7.892013,350976.583701,5.006036e+06,0.35952,-8.251533
8,Random Forest,9,"202508, 202509, 202510, 202511, 202512, 202601...",202605,0.838976,-10.985011,386612.533633,5.811833e+06,0.35952,-11.344531
2,Random Forest,3,"202602, 202603, 202604",202605,0.872692,-11.768353,459414.058827,5.998758e+06,0.35952,-12.127872
4,Decision Tree,6,"202511, 202512, 202601, 202602, 202603, 202604",202605,0.999973,-15.222809,412118.155673,6.761716e+06,0.35952,-15.582329
1,Decision Tree,3,"202602, 202603, 202604",202605,1.000000,-41.429706,542636.665221,1.093525e+07,0.35952,-41.789226


In [7]:
comparison_df["beat_baseline"] = (
    comparison_df["test_r2"] > best_baseline_r2
)

comparison_df[
    [
        "model",
        "training_window_months",
        "test_r2",
        "baseline_test_r2",
        "r2_change_from_baseline",
        "beat_baseline"
    ]
].sort_values(
    by="test_r2",
    ascending=False
)

,model,training_window_months,test_r2,baseline_test_r2,r2_change_from_baseline,beat_baseline
9,Linear Regression,12,0.359520,0.35952,0.000000,False
6,Linear Regression,9,0.350591,0.35952,-0.008929,False
3,Linear Regression,6,0.333577,0.35952,-0.025942,False
0,Linear Regression,3,0.253278,0.35952,-0.106242,False
5,Random Forest,6,-3.760875,0.35952,-4.120395,False
11,Random Forest,12,-7.892013,0.35952,-8.251533,False
8,Random Forest,9,-10.985011,0.35952,-11.344531,False
2,Random Forest,3,-11.768353,0.35952,-12.127872,False
4,Decision Tree,6,-15.222809,0.35952,-15.582329,False
1,Decision Tree,3,-41.429706,0.35952,-41.789226,False
